# Principal Component Analysis

**Companion lesson:** https://ml-viz.vercel.app/courses/pca-dimensionality/01-pca

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## PCA via Eigendecomposition

1. Center the data
2. Compute covariance matrix
3. Eigendecompose
4. Project onto top-k eigenvectors

In [ ]:
np.random.seed(42)
n = 200
t = np.linspace(0, 2 * np.pi, n)
X = np.column_stack([3 * np.cos(t), 1.5 * np.sin(t)]) + np.random.randn(n, 2) * 0.3

X_centered = X - X.mean(axis=0)
cov = np.cov(X_centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

X_pca = X_centered @ eigenvectors[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Original data with principal components
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], c='#818cf8', s=10, alpha=0.5)
mean = X.mean(axis=0)
for i in range(2):
    v = eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 2
    ax.arrow(mean[0], mean[1], v[0], v[1], head_width=0.2, head_length=0.1,
             fc='#f43f5e' if i == 0 else '#14b8a6', ec='white', linewidth=1.5)
ax.set_title('Original + PCs', color='white', fontsize=11)
ax.set_aspect('equal')

# Projected onto PC1
ax = axes[1]
ax.scatter(X_pca[:, 0], np.zeros_like(X_pca[:, 0]), c='#818cf8', s=10, alpha=0.5)
ax.set_title('Projected onto PC1', color='white', fontsize=11)
ax.set_xlabel('$z_1$')

# Explained variance
ax = axes[2]
var_ratio = eigenvalues / eigenvalues.sum()
ax.bar(range(1, len(var_ratio) + 1), var_ratio, color=['#f43f5e', '#14b8a6', '#eab308'][:len(var_ratio)])
ax.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), 'o-', color='white', linewidth=1.5)
ax.set_xlabel('Component')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot', color='white', fontsize=11)
ax.axhline(0.95, color='#94a3b8', linestyle='--', alpha=0.5, label='95% threshold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'PC1 explains {var_ratio[0]*100:.1f}% of variance')
print(f'PC1+PC2 explain {sum(var_ratio[:2])*100:.1f}% of variance')

## Explained variance and the scree plot

Each principal component captures a share of the total variance. The cumulative curve tells you how many components to keep.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

X = load_digits().data            # 64 features
pca = PCA().fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
k95 = np.argmax(cum >= 0.95) + 1
print(f'{k95} components explain 95% of variance (of {X.shape[1]})')

plt.plot(range(1, len(cum)+1), cum, color='#818cf8')
plt.axhline(0.95, ls='--', color='#f43f5e'); plt.axvline(k95, ls='--', color='#14b8a6')
plt.xlabel('components'); plt.ylabel('cumulative explained variance')
plt.title('Scree / cumulative variance'); plt.show()

## Key takeaways

- PCA finds orthogonal directions (**eigenvectors of the covariance**) of maximum variance.
- Projecting onto the top $k$ components reduces dimensions while keeping most variance.
- Choose $k$ from the **cumulative explained variance** (e.g. 95%) or the scree elbow.
- **Standardize first**; PCA is linear and unsupervised, computed efficiently via **SVD**.